In [32]:
from pathlib import Path

DATA_RAW = Path("datasets/raw")
IMAGES_DIR = "images"
LABELS_DIR = "labels"
SPLITS = ["train", "val", "test"]


# class mapping
CLASS_NAMES = {
    0: "no_animal",
    1: "red_deer",
    2: "roe_deer",
    3: "chamois",
    4: "human",
    5: "alpine_ibex",
    6: "fallow_deer",
    7: "unknown",
    8: "dog",
    9: "bird",
    10: "wild_boar",
    11: "hybrid_pig"
}

In [33]:
# Imagesize
from PIL import Image
import os

img_path = DATA_RAW / "images" / "train"

for f in os.listdir(img_path):
    if f.endswith((".jpg")):
        img = Image.open(os.path.join(img_path, f))
        print(f"name: {f}, size (w/h): {img.size}")
        break

name: 0_8082.jpg, size (w/h): (2048, 2048)


In [34]:
from pathlib import Path
from collections import Counter
import numpy as np
import csv

images_total = 0
images_with_animals = 0
images_without_animals = 0

animals_per_image = []

class_counter = Counter()

In [41]:
for split in SPLITS:
    label_dir = DATA_RAW / LABELS_DIR / split
    if not label_dir.exists():
        continue

    for file in label_dir.glob("*.txt"):
        images_total += 1

        with open(file, "r") as f:
            lines = [l.strip() for l in f if l.strip()]

        # Case: empty or only "0" → no animal image
        if len(lines) == 0 or (len(lines) == 1 and lines[0].split()[0] == "0"):
            images_without_animals += 1
            animals_per_image.append(0)
            continue

        # image contains animals
        images_with_animals += 1
        animals_per_image.append(len(lines))

        for line in lines:
            cls = int(line.split()[0])


            class_counter[cls] += 1

total_animals = sum(class_counter.values())
avg_animals_per_image = total_animals / images_total if images_total else 0

animals_array = np.array(animals_per_image)

In [42]:
print(f"Total images: {images_total}")
print(f"Images with animals: {images_with_animals}")
print(f"Images without animals: {images_without_animals}")
print(f"Percentage with animals: {images_with_animals / images_total * 100:.2f}%")

print("\n--- Animal stats ---")
print(f"Total animals: {total_animals}")
print(f"Average animals per image: {avg_animals_per_image:.2f}")
print(f"Max animals in one image: {animals_array.max()}")

print(f"95th percentile animals/image: {np.percentile(animals_array, 95):.2f}")
print(f"99th percentile animals/image: {np.percentile(animals_array, 99):.2f}")

print("\n--- Per class distribution ---")

all_class_ids = sorted(CLASS_NAMES.keys())

for cls_id in all_class_ids:
    name = CLASS_NAMES[cls_id]
    count = class_counter.get(cls_id, 0)

    print(f"{cls_id:2d} {name:15s}: {count}")


Total images: 38168
Images with animals: 36708
Images without animals: 1460
Percentage with animals: 96.17%

--- Animal stats ---
Total animals: 140197
Average animals per image: 3.67
Max animals in one image: 69
95th percentile animals/image: 12.00
99th percentile animals/image: 36.00

--- Per class distribution ---
 0 no_animal      : 1647
 1 red_deer       : 55392
 2 roe_deer       : 7230
 3 chamois        : 10222
 4 human          : 47372
 5 alpine_ibex    : 6710
 6 fallow_deer    : 7558
 7 unknown        : 2374
 8 dog            : 1692
 9 bird           : 0
10 wild_boar      : 0
11 hybrid_pig     : 0


In [43]:
print("\n--- Class imbalance (percentage, FULL) ---")

all_class_ids = sorted(CLASS_NAMES.keys())

for cls_id in all_class_ids:
    count = class_counter.get(cls_id, 0)
    pct = (count / total_animals) * 100 if total_animals else 0

    print(f"{CLASS_NAMES[cls_id]:15s}: {pct:.2f}%")


--- Class imbalance (percentage, FULL) ---
no_animal      : 1.17%
red_deer       : 39.51%
roe_deer       : 5.16%
chamois        : 7.29%
human          : 33.79%
alpine_ibex    : 4.79%
fallow_deer    : 5.39%
unknown        : 1.69%
dog            : 1.21%
bird           : 0.00%
wild_boar      : 0.00%
hybrid_pig     : 0.00%


In [44]:
print("\n--- Rare classes (<5%) ---")

for cls_id in all_class_ids:
    count = class_counter.get(cls_id, 0)
    pct = (count / total_animals) * 100 if total_animals else 0

    if pct < 5:
        print(f"{CLASS_NAMES[cls_id]:15s}: {count:5d} - {pct:.2f}%")


--- Rare classes (<5%) ---
no_animal      :  1647 - 1.17%
alpine_ibex    :  6710 - 4.79%
unknown        :  2374 - 1.69%
dog            :  1692 - 1.21%
bird           :     0 - 0.00%
wild_boar      :     0 - 0.00%
hybrid_pig     :     0 - 0.00%


In [45]:
from collections import Counter as C
dist = C(animals_per_image)

print("--- Animals per image distribution ---")
for k in sorted(dist):
    print(f"{k} animals: {dist[k]} images")

--- Animals per image distribution ---
0 animals: 1460 images
1 animals: 16006 images
2 animals: 7670 images
3 animals: 3522 images
4 animals: 1956 images
5 animals: 1360 images
6 animals: 950 images
7 animals: 914 images
8 animals: 774 images
9 animals: 586 images
10 animals: 396 images
11 animals: 322 images
12 animals: 350 images
13 animals: 296 images
14 animals: 182 images
15 animals: 136 images
16 animals: 146 images
17 animals: 74 images
18 animals: 116 images
19 animals: 66 images
20 animals: 24 images
21 animals: 26 images
22 animals: 24 images
23 animals: 40 images
24 animals: 50 images
25 animals: 38 images
26 animals: 14 images
27 animals: 16 images
28 animals: 24 images
29 animals: 8 images
30 animals: 6 images
31 animals: 12 images
32 animals: 24 images
33 animals: 36 images
34 animals: 50 images
35 animals: 102 images
36 animals: 56 images
37 animals: 6 images
38 animals: 4 images
39 animals: 6 images
40 animals: 8 images
42 animals: 6 images
43 animals: 10 images
44 ani

In [46]:
with open("analysis/dataset_class_distribution.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["class_id", "class_name", "count", "percentage"])

    for cls_id, count in sorted(class_counter.items()):
        pct = (count / total_animals) * 100 if total_animals else 0
        writer.writerow([cls_id, CLASS_NAMES.get(cls_id), count, pct])

print("\nCSV saved: dataset_class_distribution.csv")


CSV saved: dataset_class_distribution.csv


In [ ]:
# look at black and blurred images -> count:
from pathlib import Path

def scan_quality_stats(base_path: Path, splits, images_dir="images"):
    stats = {}

    for split in splits:
        img_dir = base_path / images_dir / split

        total = 0
        black = 0
        blurry = 0

        for img_path in img_dir.glob("*.jpg"):
            total += 1

            if is_mostly_black(img_path):
                black += 1
                continue

            if is_blurry(img_path):
                blurry += 1
                continue

        stats[split] = {
            "total": total,
            "black": black,
            "blurry": blurry,
            "kept": total - black - blurry
        }

    return stats

In [47]:
from utils.exploring import is_mostly_black, is_blurry

from pathlib import Path

def scan_quality_stats(base_path: Path, splits, images_dir="images"):
    stats = {}

    for split in splits:
        img_dir = base_path / images_dir / split

        total = 0
        black = 0
        blurry = 0

        for img_path in img_dir.glob("*.jpg"):
            total += 1

            if is_mostly_black(img_path):
                black += 1
                continue

            if is_blurry(img_path):
                blurry += 1
                continue

        stats[split] = {
            "total": total,
            "black": black,
            "blurry": blurry,
            "kept": total - black - blurry
        }

    return stats

In [ ]:
raw_stats = scan_quality_stats(DATA_RAW, SPLITS)

print("\n===== QUALITY CHECK (RAW DATA) =====")
for split, s in raw_stats.items():
    print(
        f"{split.upper():5s} | "
        f"total={s['total']:5d} | "
        f"kept={s['kept']:5d} | "
        f"black={s['black']:5d} | "
        f"blurry={s['blurry']:5d}"
    )